In [5]:
!pip install langchain chromadb langchain_huggingface huggingface-hub tiktoken pypdf langchain-community langchain_chroma

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [8]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [9]:
vector_store = Chroma(
    embedding_function=embedding,
    persist_directory='my_chroma_db',
    collection_name='sample'
)

In [10]:
# add documents
vector_store.add_documents(docs)

['2861ed13-c60c-4f0e-9961-1f41c62a6cab',
 '9fc34915-d3ac-4ffa-b21e-f3e02e58ce7f',
 'cc8439fe-7087-4f75-b62d-faf1a151615e',
 '9b47aa29-1e2a-4f3d-bfd7-cfa96946e838',
 '91959073-1685-42bb-b570-f59252e4f684']

In [11]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['2861ed13-c60c-4f0e-9961-1f41c62a6cab',
  '9fc34915-d3ac-4ffa-b21e-f3e02e58ce7f',
  'cc8439fe-7087-4f75-b62d-faf1a151615e',
  '9b47aa29-1e2a-4f3d-bfd7-cfa96946e838',
  '91959073-1685-42bb-b570-f59252e4f684'],
 'embeddings': array([[ 0.00994728,  0.06914336, -0.05147117, ..., -0.03543339,
          0.01284808,  0.01248293],
        [ 0.00127746,  0.03129853, -0.02375378, ..., -0.0051836 ,
         -0.03280611,  0.02737715],
        [-0.10265916,  0.02650813,  0.02271503, ..., -0.03359744,
         -0.07984944, -0.01507706],
        [ 0.02123395, -0.02468549, -0.04494376, ..., -0.10995813,
          0.00572561,  0.09915381],
        [ 0.0187398 ,  0.04382842, -0.04304253, ..., -0.0780162 ,
         -0.07840683, -0.00304189]]),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful ca

In [12]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(id='9b47aa29-1e2a-4f3d-bfd7-cfa96946e838', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='9fc34915-d3ac-4ffa-b21e-f3e02e58ce7f', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.")]

In [13]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='9b47aa29-1e2a-4f3d-bfd7-cfa96946e838', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9693599343299866),
 (Document(id='9fc34915-d3ac-4ffa-b21e-f3e02e58ce7f', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  1.149344801902771)]

In [14]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="",
    filter={"team": "Chennai Super Kings"}
)

[(Document(id='cc8439fe-7087-4f75-b62d-faf1a151615e', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.8436005115509033),
 (Document(id='91959073-1685-42bb-b570-f59252e4f684', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.890937328338623)]

In [15]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='09a39dc6-3ba6-4ea7-927e-fdda591da5e4', document=updated_doc1)


In [16]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['2861ed13-c60c-4f0e-9961-1f41c62a6cab',
  '9fc34915-d3ac-4ffa-b21e-f3e02e58ce7f',
  'cc8439fe-7087-4f75-b62d-faf1a151615e',
  '9b47aa29-1e2a-4f3d-bfd7-cfa96946e838',
  '91959073-1685-42bb-b570-f59252e4f684'],
 'embeddings': array([[ 0.00994728,  0.06914336, -0.05147117, ..., -0.03543339,
          0.01284808,  0.01248293],
        [ 0.00127746,  0.03129853, -0.02375378, ..., -0.0051836 ,
         -0.03280611,  0.02737715],
        [-0.10265916,  0.02650813,  0.02271503, ..., -0.03359744,
         -0.07984944, -0.01507706],
        [ 0.02123395, -0.02468549, -0.04494376, ..., -0.10995813,
          0.00572561,  0.09915381],
        [ 0.0187398 ,  0.04382842, -0.04304253, ..., -0.0780162 ,
         -0.07840683, -0.00304189]]),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful ca

In [17]:
# delete document
vector_store.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])

In [18]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['2861ed13-c60c-4f0e-9961-1f41c62a6cab',
  '9fc34915-d3ac-4ffa-b21e-f3e02e58ce7f',
  'cc8439fe-7087-4f75-b62d-faf1a151615e',
  '9b47aa29-1e2a-4f3d-bfd7-cfa96946e838',
  '91959073-1685-42bb-b570-f59252e4f684'],
 'embeddings': array([[ 0.00994728,  0.06914336, -0.05147117, ..., -0.03543339,
          0.01284808,  0.01248293],
        [ 0.00127746,  0.03129853, -0.02375378, ..., -0.0051836 ,
         -0.03280611,  0.02737715],
        [-0.10265916,  0.02650813,  0.02271503, ..., -0.03359744,
         -0.07984944, -0.01507706],
        [ 0.02123395, -0.02468549, -0.04494376, ..., -0.10995813,
          0.00572561,  0.09915381],
        [ 0.0187398 ,  0.04382842, -0.04304253, ..., -0.0780162 ,
         -0.07840683, -0.00304189]]),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful ca